# Maps

Annual-mean maps from the HAMOCC annual history files (`hbgcy`): surface fields and depth slices, shown as model | observations | model - observations. Observations are interpolated onto the model grid.

In [ ]:
import matplotlib.pyplot as plt
import yaml
from pyprojroot import here

from blomster import grid
from hamocc import climatology, io, obs, plots, timeseries, variables

In [ ]:
# Depth slices to show (m); must match levels of the model's z-level fields.
depths = list(variables.DEPTH_LEVELS_M)
compare_obs = True

In [ ]:
with open(here('notebooks_hamocc/settings.yml')) as f:
    cfg = yaml.safe_load(f)
exp = cfg['experiment']
dsets = cfg['datasets']

lat, lon, _, clat, clon, _ = grid.load(cfg['grids'][exp['grid']]['grid_file'])

names = [n for n in list(variables.SURFACE_VARS) + ['co2fxn'] + list(variables.LEVEL_VARS)]
needed = sorted({h for n in names for h in climatology.history_names(n)} | {'depth_bnds'})
files = io.history_files(exp['input_root'], exp['caseid'], 'hbgcy', exp['year_start'], exp['year_end'])
ds = io.open_history(files, [n for n in needed if n != 'depth_bnds'])

parea = timeseries.load_parea(cfg['grids'][exp['grid']]['grid_file'])

def obs_on_model(name, depth=None):
    spec = dsets['maps'].get(name)
    if not compare_obs or spec is None:
        return None
    return obs.regrid_to_model(obs.load_obs(spec, dsets['obs_root'], depth), lat, lon)

def show(name, depth=None):
    model = climatology.annual_mean_map(ds, name, depth).load()
    title = model.attrs['long_name'] + (f', {depth} m' if depth is not None else '')
    fig = plots.map_panels(clon, clat, model, obs_on_model(name, depth), title=title, area=parea)
    plt.show()

```{note}
Observation units and scale factors in `recipe_hamocc.yml` are not yet verified against the files (`datasets.units_verified: false`). Check the difference panels before interpreting them quantitatively.
```

## Surface fields

In [ ]:
for name in list(variables.SURFACE_VARS) + ['co2fxn']:
    show(name)

## Depth slices

In [ ]:
for name in variables.LEVEL_VARS:
    for depth in depths:
        show(name, depth)